# EmpathBot_V1 — Modified ResNet18

**Team 14 · CS731** | Preeti · TJ · Kanishka

Trains EmpathBot_V1 (ResNet18 + SE-attention + BN head) on the team dataset, then fine-tunes the head on Kanishka's manually-labelled personal images (`kash_dataset`).

| Phase | Dataset | What trains |
|---|---|---|
| 1 — Main training | `preetiv1/dataset-facial` via `master_split.csv` | Full model (backbone frozen first 5 ep, then unfrozen) |
| 2 — Personal fine-tune | `kash_dataset` (manually labelled) | Head only (backbone stays frozen) |

**Architecture changes over vanilla ResNet18**
| What | Vanilla | EmpathBot_V1 |
|---|---|---|
| Channel attention | None | SE block after each of the 4 residual stages |
| Classifier | `Linear(512→6)` | `Linear→BN→ReLU→Dropout(0.4)→Linear→BN→ReLU→Dropout(0.2)→Linear` |
| Optimizer | Adam/SGD | AdamW, differential LR (backbone 1e-4, head 1e-3) |
| Schedule | StepLR | 3-ep linear warmup → CosineAnnealingLR |
| Augmentation | Uniform | Class-specific (harder for sadness / fear_anxiety / distrust) |

## 1. Imports & reproducibility

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import classification_report, confusion_matrix, recall_score
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Paths & config

In [ ]:
# ── Dataset (team Kaggle dataset) ─────────────────────────────────────────────
BASE_DIR   = Path('/kaggle/input/datasets/preetiv1/dataset-facial')
MASTER_CSV = BASE_DIR / 'data' / 'details' / 'master_split.csv'

# ── kash_dataset: Kanishka's manually-labelled personal images ────────────────
# Expected layout — two options, use whichever matches your upload:
#
#   OPTION A (folder per class — recommended):
#     /kaggle/input/kash-dataset/
#         neutral/   trust_relief/   sadness/   fear_anxiety/   confusion/   distrust/
#
#   OPTION B (flat folder + CSV):
#     /kaggle/input/kash-dataset/images/   ← all images here
#     /kaggle/input/kash-dataset/labels.csv  ← columns: filename, eb_label (0-5)
#
KASH_DIR    = Path('/kaggle/input/kash-dataset')   # <── change to your dataset slug
KASH_OPTION = 'A'   # 'A' = folder-per-class  |  'B' = flat + CSV

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = Path('/kaggle/working/empathbot_v1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── EmpathBot 6-class label map ───────────────────────────────────────────────
CLASS_NAMES = {
    0: 'neutral',
    1: 'trust_relief',
    2: 'sadness',
    3: 'fear_anxiety',
    4: 'confusion',
    5: 'distrust',
}
NUM_CLASSES = len(CLASS_NAMES)   # 6

# ── Training hyperparameters ──────────────────────────────────────────────────
CFG = dict(
    backbone        = 'resnet18',  # 'resnet18' or 'efficientnet_b0'
    se_reduction    = 16,

    epochs          = 25,
    batch_size      = 64,
    img_size        = 224,
    freeze_epochs   = 5,           # backbone frozen for first N epochs

    backbone_lr     = 1e-4,
    head_lr         = 1e-3,
    weight_decay    = 1e-4,

    warmup_epochs   = 3,
    min_lr          = 1e-6,

    label_smoothing = 0.05,
    mixup_alpha     = 0.2,
    grad_clip       = 1.0,
    patience        = 8,

    # sadness + fear_anxiety are the 'angry/sad' analogues in EmpathBot class space
    priority_classes = [2, 3],     # eb_label IDs for sadness, fear_anxiety

    num_workers     = 2,
)

print('Config ready.')
print(f'Classes: {list(CLASS_NAMES.values())}')
print(f'Priority recall: {[CLASS_NAMES[i] for i in CFG["priority_classes"]]}')

## 3. Load master_split.csv and inspect class distribution

In [ ]:
df = pd.read_csv(MASTER_CSV)
print(f'master_split.csv: {len(df):,} rows | columns: {list(df.columns)}')

train_df = df[df['split'] == 'train'].copy().reset_index(drop=True)
val_df   = df[df['split'] == 'val'].copy().reset_index(drop=True)
test_df  = df[df['split'] == 'test'].copy().reset_index(drop=True)

print(f'\nSplit sizes:  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

print('\nClass distribution (train):')
counts = np.zeros(NUM_CLASSES, dtype=int)
for label_id, name in CLASS_NAMES.items():
    n = (train_df['eb_label'] == label_id).sum()
    counts[label_id] = n
    bar = '█' * (n // 200)
    print(f'  {label_id}  {name:<15}: {n:>5}  {bar}')

# Sanity-check path column
sample_path = Path(train_df.iloc[0]['path'])
print(f'\nSample path : {sample_path}')
print(f'File exists : {sample_path.exists()}')
if not sample_path.exists():
    print('⚠️  Path not found — check that BASE_DIR is correct and paths in CSV are absolute.')

df.head(3)

## 4. Transforms & Dataset class

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
SZ   = CFG['img_size']

# Minority/priority classes get stronger augmentation
HARD_LABEL_IDS = set(CFG['priority_classes']) | {5}  # sadness, fear_anxiety, distrust

BASE_AUG = T.Compose([
    T.Resize((SZ + 32, SZ + 32)),
    T.RandomCrop(SZ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

STRONG_AUG = T.Compose([
    T.Resize((SZ + 32, SZ + 32)),
    T.RandomCrop(SZ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomRotation(15),
    T.RandomGrayscale(p=0.1),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

VAL_TF = T.Compose([
    T.Resize((SZ, SZ)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


class EmpathBotDataset(Dataset):
    """
    Reads from master_split.csv (columns: path, eb_label, split).
    Applies class-specific augmentation: harder transforms for hard label IDs.
    Missing files are silently skipped.
    """

    def __init__(self, dataframe: pd.DataFrame, hard_ids: set, is_train: bool):
        valid = dataframe['path'].apply(lambda p: Path(p).exists())
        n_missing = (~valid).sum()
        if n_missing:
            print(f'  ⚠️  {n_missing} missing files skipped')
        self.df       = dataframe[valid].reset_index(drop=True)
        self.hard_ids = hard_ids
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['path']).convert('RGB')
        label = int(row['eb_label'])
        if self.is_train:
            tf = STRONG_AUG if label in self.hard_ids else BASE_AUG
        else:
            tf = VAL_TF
        return tf(img), label


print('Building datasets...')
train_ds = EmpathBotDataset(train_df, HARD_LABEL_IDS, is_train=True)
val_ds   = EmpathBotDataset(val_df,   HARD_LABEL_IDS, is_train=False)
test_ds  = EmpathBotDataset(test_df,  HARD_LABEL_IDS, is_train=False)
print(f'  train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

In [ ]:
# ── Class weights (inverse frequency + 1.3× boost for hard classes) ───────────
train_labels = train_ds.df['eb_label'].values.astype(int)
cls_counts   = np.bincount(train_labels, minlength=NUM_CLASSES).astype(float)
cls_weights  = 1.0 / np.where(cls_counts == 0, 1.0, cls_counts)
for hid in HARD_LABEL_IDS:
    cls_weights[hid] *= 1.3
class_weights_t = torch.tensor(cls_weights, dtype=torch.float32).to(DEVICE)

print('Class weights for CrossEntropyLoss:')
for i, name in CLASS_NAMES.items():
    print(f'  {i}  {name:<15}  n={int(cls_counts[i]):>5}  w={cls_weights[i]:.5f}')

# WeightedRandomSampler — balanced batches
sample_w = cls_weights[train_labels]
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

## 5. EmpathBot_V1 architecture

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (Hu et al. 2018)."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c = x.shape[:2]
        s = self.avg(x).view(b, c)
        return x * self.fc(s).view(b, c, 1, 1)


def _make_head(in_features: int, num_classes: int) -> nn.Sequential:
    mid = max(in_features // 2, 256)
    return nn.Sequential(
        nn.Linear(in_features, mid),
        nn.BatchNorm1d(mid),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(mid, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(128, num_classes),
    )


class EmpathBotV1(nn.Module):
    """
    ResNet18 + SE-attention after each residual stage + 3-layer BN classifier head.
    Switchable to EfficientNet-B0 via backbone='efficientnet_b0'.
    """

    _FEAT_DIMS = {'resnet18': 512, 'efficientnet_b0': 1280}

    def __init__(self, num_classes: int, backbone: str = 'resnet18', se_reduction: int = 16):
        super().__init__()
        assert backbone in self._FEAT_DIMS
        self.backbone_name = backbone

        if backbone == 'resnet18':
            b = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.stem   = nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool)
            self.layer1 = b.layer1 ; self.se1 = SEBlock(64,  se_reduction)
            self.layer2 = b.layer2 ; self.se2 = SEBlock(128, se_reduction)
            self.layer3 = b.layer3 ; self.se3 = SEBlock(256, se_reduction)
            self.layer4 = b.layer4 ; self.se4 = SEBlock(512, se_reduction)
        else:
            b = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.features = b.features

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = _make_head(self._FEAT_DIMS[backbone], num_classes)

    def forward(self, x):
        if self.backbone_name == 'resnet18':
            x = self.stem(x)
            x = self.se1(self.layer1(x))
            x = self.se2(self.layer2(x))
            x = self.se3(self.layer3(x))
            x = self.se4(self.layer4(x))
        else:
            x = self.features(x)
        return self.head(self.pool(x).flatten(1))

    def backbone_params(self):
        if self.backbone_name == 'resnet18':
            return [
                *self.stem.parameters(),
                *self.layer1.parameters(), *self.se1.parameters(),
                *self.layer2.parameters(), *self.se2.parameters(),
                *self.layer3.parameters(), *self.se3.parameters(),
                *self.layer4.parameters(), *self.se4.parameters(),
            ]
        return list(self.features.parameters())

    def head_params(self):
        return list(self.head.parameters())


model = EmpathBotV1(
    num_classes=NUM_CLASSES,
    backbone=CFG['backbone'],
    se_reduction=CFG['se_reduction'],
).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f'EmpathBot_V1 ({CFG["backbone"]}) — {total/1e6:.2f}M total params')

## 6. Optimiser, loss, scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights_t,
    label_smoothing=CFG['label_smoothing'],
)

# Backbone gets 10× smaller LR than fresh head (standard transfer-learning practice)
optimizer = optim.AdamW(
    [
        {'params': model.backbone_params(), 'lr': CFG['backbone_lr']},
        {'params': model.head_params(),     'lr': CFG['head_lr']},
    ],
    weight_decay=CFG['weight_decay'],
)

# Linear warmup for warmup_epochs then cosine decay to min_lr
steps_per_epoch = len(train_loader)
warmup_steps    = CFG['warmup_epochs'] * steps_per_epoch
total_steps     = CFG['epochs']        * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    cosine   = 0.5 * (1.0 + np.cos(np.pi * progress))
    floor    = CFG['min_lr'] / CFG['head_lr']
    return floor + (1.0 - floor) * cosine

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'AdamW  backbone_lr={CFG["backbone_lr"]}  head_lr={CFG["head_lr"]}')
print(f'Schedule: {CFG["warmup_epochs"]}-ep warmup → CosineAnnealingLR')
print(f'CrossEntropyLoss  label_smoothing={CFG["label_smoothing"]}  class_weights=yes')

## 7. Training

In [ ]:
def mixup(x, y, alpha):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def train_epoch(model, loader, optimizer, scheduler, criterion, freeze_backbone):
    model.train()
    for p in model.backbone_params():
        p.requires_grad_(not freeze_backbone)

    loss_sum, correct, n = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        if CFG['mixup_alpha'] > 0:
            imgs, ya, yb, lam = mixup(imgs, labels, CFG['mixup_alpha'])
            logits = model(imgs)
            loss   = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
        else:
            logits = model(imgs)
            loss   = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        optimizer.step()
        scheduler.step()

        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        n        += labels.size(0)

    return loss_sum / n, correct / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds_all, labels_all = [], []
    for imgs, labels in loader:
        preds_all.extend(model(imgs.to(DEVICE)).argmax(1).cpu().tolist())
        labels_all.extend(labels.tolist())
    preds_all  = np.array(preds_all)
    labels_all = np.array(labels_all)
    acc           = (preds_all == labels_all).mean()
    per_cls_recall = recall_score(labels_all, preds_all, average=None,
                                  labels=list(range(NUM_CLASSES)), zero_division=0)
    return acc, per_cls_recall, preds_all, labels_all


# ── Training loop ─────────────────────────────────────────────────────────────
history = {'train_loss': [], 'train_acc': [], 'val_acc': [],
           'sadness_recall': [], 'fear_anxiety_recall': []}
best_val_acc, patience_cnt = 0.0, 0
best_ckpt = OUT_DIR / 'best.pth'

print(f'Training {CFG["epochs"]} epochs | batch={CFG["batch_size"]} | '
      f'backbone frozen first {CFG["freeze_epochs"]} eps')
print(f'Priority recall: sadness (2) + fear_anxiety (3)\n')
print(f'{"Ep":>4}  {"loss":>8}  {"train":>6}  {"val":>6}  {"sadness":>8}  {"fear_anx":>8}')
print('-' * 60)

for epoch in range(1, CFG['epochs'] + 1):
    frozen = epoch <= CFG['freeze_epochs']
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, frozen)
    val_acc, per_cls, _, _ = evaluate(model, val_loader)

    sad_rec  = per_cls[2]  # sadness
    fear_rec = per_cls[3]  # fear_anxiety

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['sadness_recall'].append(sad_rec)
    history['fear_anxiety_recall'].append(fear_rec)

    flag = '  [frozen]' if frozen else ''
    print(f'{epoch:4d}  {tr_loss:8.4f}  {tr_acc:6.3f}  {val_acc:6.3f}  '
          f'{sad_rec:8.3f}  {fear_rec:8.3f}{flag}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_cnt = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'val_acc': val_acc,
            'per_cls_recall': per_cls.tolist(),
            'class_names': CLASS_NAMES,
            'cfg': CFG,
        }, best_ckpt)
        print(f'       ↑ new best {val_acc:.4f} saved')
    else:
        patience_cnt += 1
        if patience_cnt >= CFG['patience']:
            print(f'\nEarly stop at epoch {epoch}')
            break

print(f'\nBest val accuracy: {best_val_acc:.4f}')

## 8. Evaluation on test set

In [ ]:
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint — epoch {ckpt["epoch"]}  val_acc={ckpt["val_acc"]:.4f}\n')

test_acc, test_per_cls, test_preds, test_labels = evaluate(model, test_loader)
class_name_list = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]

print(classification_report(test_labels, test_preds, target_names=class_name_list, digits=3))
print(f'Priority class recall:')
print(f'  sadness      : {test_per_cls[2]:.3f}')
print(f'  fear_anxiety : {test_per_cls[3]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Per-class recall bar chart
ax = axes[0]
colors = ['coral' if i in CFG['priority_classes'] else 'steelblue' for i in range(NUM_CLASSES)]
bars = ax.bar(class_name_list, test_per_cls, color=colors, alpha=0.85)
ax.axhline(0.7, color='gray', linestyle='--', alpha=0.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Recall')
ax.set_title('Per-class Recall on Test Set (orange = priority)')
ax.tick_params(axis='x', rotation=30)
for bar, v in zip(bars, test_per_cls):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_name_list, yticklabels=class_name_list, ax=axes[1])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_title('Normalised Confusion Matrix — EmpathBot_V1')

plt.tight_layout()
plt.savefig(OUT_DIR / 'eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training curves
epochs_ran = range(1, len(history['train_acc']) + 1)
fig, axes  = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(epochs_ran, history['train_loss'])
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Train Loss')

axes[1].plot(epochs_ran, history['train_acc'], label='Train')
axes[1].plot(epochs_ran, history['val_acc'],   label='Val')
axes[1].axvline(CFG['freeze_epochs'], color='gray', linestyle='--', alpha=0.5, label='Unfreeze')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Accuracy')
axes[1].legend()

axes[2].plot(epochs_ran, history['sadness_recall'],      label='sadness')
axes[2].plot(epochs_ran, history['fear_anxiety_recall'], label='fear_anxiety')
axes[2].set(xlabel='Epoch', ylabel='Recall', title='Priority Class Recall')
axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. kash_dataset — fine-tune head on Kanishka's personal images

The backbone stays fully frozen. Only the 3-layer classifier head is updated.
This adapts the model to Kanishka's face without overfitting on a tiny dataset.

**How to set up the dataset on Kaggle:**

Since you labelled the images manually (no folder structure), create a CSV:
```
filename,eb_label,split
img_001.jpg,0,train
img_002.jpg,2,train
img_003.jpg,2,val
...
```
where `eb_label` is 0–5 matching the 6 EmpathBot classes above.
Upload both the images folder and this CSV as a single Kaggle dataset.

Set `KASH_DIR` and `KASH_CSV` below to match your upload.

In [ ]:
KASH_DIR = Path('/kaggle/input/kash-dataset')   # <── your Kaggle dataset slug
KASH_IMG_DIR = KASH_DIR / 'images'              # folder containing all images
KASH_CSV     = KASH_DIR / 'labels.csv'          # CSV with columns: filename, eb_label, split

kash_df = pd.read_csv(KASH_CSV)
# Absolute paths
kash_df['path'] = kash_df['filename'].apply(lambda f: str(KASH_IMG_DIR / f))

kash_train_df = kash_df[kash_df['split'] == 'train'].reset_index(drop=True)
kash_val_df   = kash_df[kash_df['split'] == 'val'].reset_index(drop=True)

print(f'kash_dataset — train: {len(kash_train_df)} | val: {len(kash_val_df)}')
print('\nClass distribution:')
for i, name in CLASS_NAMES.items():
    n = (kash_train_df['eb_label'] == i).sum()
    print(f'  {i}  {name:<15}: {n}')

In [ ]:
# Aggressive augmentation for tiny personal dataset
KASH_AUG = T.Compose([
    T.Resize((SZ + 40, SZ + 40)),
    T.RandomCrop(SZ),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.08),
    T.RandomRotation(15),
    T.RandomGrayscale(p=0.15),
    T.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.88, 1.12)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

kash_train_ds = EmpathBotDataset(kash_train_df, set(), is_train=False)  # is_train=False → we set tf manually
kash_val_ds   = EmpathBotDataset(kash_val_df,   set(), is_train=False)

# Override transform with the aggressive aug for train
class KashDataset(Dataset):
    def __init__(self, df, transform):
        valid = df['path'].apply(lambda p: Path(p).exists())
        self.df = df[valid].reset_index(drop=True)
        self.tf = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.tf(Image.open(row['path']).convert('RGB')), int(row['eb_label'])

kash_train_ds = KashDataset(kash_train_df, KASH_AUG)
kash_val_ds   = KashDataset(kash_val_df,   VAL_TF)

kash_train_loader = DataLoader(kash_train_ds, batch_size=8,  shuffle=True,  num_workers=2)
kash_val_loader   = DataLoader(kash_val_ds,   batch_size=8,  shuffle=False, num_workers=2)

print(f'Loaders ready — train batches: {len(kash_train_loader)}')

In [ ]:
# Build personalised model: same architecture, load trained backbone, replace + train head only
kash_model = EmpathBotV1(
    num_classes=NUM_CLASSES,
    backbone=CFG['backbone'],
    se_reduction=CFG['se_reduction'],
).to(DEVICE)
kash_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE)['model_state'])

# Freeze backbone permanently for fine-tuning
for p in kash_model.backbone_params():
    p.requires_grad_(False)

# Fresh head (same 6 classes — backbone already speaks EmpathBot label space)
feat_dim = EmpathBotV1._FEAT_DIMS[CFG['backbone']]
kash_model.head = _make_head(feat_dim, NUM_CLASSES).to(DEVICE)

trainable = sum(p.numel() for p in kash_model.parameters() if p.requires_grad)
print(f'Trainable (head only): {trainable/1e6:.3f}M params')

# Class weights from kash distribution
kash_counts  = np.bincount(kash_train_df['eb_label'].values.astype(int), minlength=NUM_CLASSES).astype(float)
kash_weights = torch.tensor(
    1.0 / np.where(kash_counts == 0, 1.0, kash_counts), dtype=torch.float32
).to(DEVICE)

kash_crit  = nn.CrossEntropyLoss(weight=kash_weights, label_smoothing=0.05)
kash_optim = optim.AdamW(kash_model.head.parameters(), lr=5e-4, weight_decay=1e-4)

KASH_EPOCHS = 30
kash_sched  = optim.lr_scheduler.CosineAnnealingLR(kash_optim, T_max=KASH_EPOCHS, eta_min=1e-6)

best_kash_acc = 0.0
kash_ckpt     = OUT_DIR / 'kash_best.pth'

print(f'\nFine-tuning head on kash_dataset for {KASH_EPOCHS} epochs\n')
print(f'{"Ep":>4}  {"loss":>8}  {"train":>6}  {"val":>6}')
print('-' * 36)

for epoch in range(1, KASH_EPOCHS + 1):
    kash_model.train()
    ls, corr, n = 0.0, 0, 0
    for imgs, labels in kash_train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = kash_model(imgs)
        loss   = kash_crit(logits, labels)
        kash_optim.zero_grad(); loss.backward(); kash_optim.step()
        ls   += loss.item() * imgs.size(0)
        corr += (logits.argmax(1) == labels).sum().item()
        n    += labels.size(0)
    kash_sched.step()

    k_acc, k_per_cls, _, _ = evaluate(kash_model, kash_val_loader)
    print(f'{epoch:4d}  {ls/n:8.4f}  {corr/n:6.3f}  {k_acc:6.3f}')

    if k_acc > best_kash_acc:
        best_kash_acc = k_acc
        torch.save({'epoch': epoch, 'model_state': kash_model.state_dict(),
                    'val_acc': k_acc, 'class_names': CLASS_NAMES}, kash_ckpt)
        print(f'       ↑ new best {k_acc:.4f}')

print(f'\nkash fine-tuning done. Best val acc: {best_kash_acc:.4f}')

In [ ]:
kash_model.load_state_dict(torch.load(kash_ckpt, map_location=DEVICE)['model_state'])
_, k_per_cls, k_preds, k_labels = evaluate(kash_model, kash_val_loader)

print(classification_report(k_labels, k_preds, target_names=class_name_list, digits=3))

cm_k = confusion_matrix(k_labels, k_preds, normalize='true')
plt.figure(figsize=(7, 5))
sns.heatmap(cm_k, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_name_list, yticklabels=class_name_list)
plt.title('kash_dataset — Confusion Matrix (head fine-tuned)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig(OUT_DIR / 'kash_cm.png', dpi=150)
plt.show()